# 🧮 IA Calculadora — Investigación de Operaciones

Sistema inteligente que combina solvers numéricos (scipy, PuLP, Dijkstra) con RAG sobre Gemini para resolver problemas de IO.

**Arquitectura modular:**
- `config.py` — Configuración centralizada
- `embeddings.py` — ChromaDB + generación de embeddings
- `solvers.py` — Solucionadores numéricos
- `rag.py` — Consultas a Gemini con contexto del libro

In [1]:
# ============================================================
# 1. IMPORTS Y CARGA DEL SISTEMA
# ============================================================
import sys
import os

# Asegurar que los módulos se encuentren
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from config import GOOGLE_API_KEY
from embeddings import load_or_create_vectorstore
from rag import preguntar_io

print(f"✅ Módulos cargados")
print(f"✅ API Key configurada (termina en ...{GOOGLE_API_KEY[-4:]})")

In [2]:
# ============================================================
# 2. CARGAR VECTOR STORE (ChromaDB)
# Primera vez: genera embeddings desde el PDF (~5 min)
# Siguientes veces: carga instantáneamente (0 API calls)
# ============================================================

vectorstore, embeddings_model = load_or_create_vectorstore()

INFO     | io_calc.embeddings   | === Iniciando carga/creación de vectorstore ===
INFO     | io_calc.embeddings   | Inicializando modelo de embeddings: models/gemini-embedding-001
INFO     | io_calc.embeddings   | Cargando PDF desde d:\IA - Calculadora\src\..\Insumos\Investigacion-Operaciones10Edicion-Frederick-S-Hillier.pdf
INFO     | io_calc.embeddings   | PDF cargado: 1229 páginas
INFO     | io_calc.embeddings   | 2996 fragmentos creados (chunk_size=2000, overlap=400)
INFO     | io_calc.embeddings   | Total de chunks deseados: 2996
INFO     | io_calc.embeddings   | ChromaDB encontrado en d:\IA - Calculadora\src\..\chroma_db
d:\IA - Calculadora\src\embeddings.py:150: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector

⚡ Vector store completo: 2996 fragmentos (0 API calls)


In [3]:
# ==================== EJEMPLOS DE USO ====================

print("\n" + "="*60)
print("EJEMPLO 1: Programación Lineal con RAG (Gemini)")
print("="*60)

pregunta_lp = """
Maximizar 40x1 + 30x2
Sujeto a:
2x1 + x2 <= 100 (horas máquina)
x1 + 2x2 <= 80 (horas mano de obra)
x1, x2 >= 0
"""

resultado_lp = preguntar_io(pregunta_lp, vectorstore)
print(resultado_lp)

In [4]:
print("\n" + "="*60)
print("EJEMPLO 2: Programación Lineal Entera con RAG (Gemini)")
print("="*60)

pregunta_ip = """
Maximizar 40x1 + 30x2
Sujeto a:
2x1 + x2 <= 100 (horas máquina)
x1 + 2x2 <= 80 (horas mano de obra)
x1, x2 deben ser enteros no negativos.
"""

resultado_ip = preguntar_io(pregunta_ip, vectorstore)
print(resultado_ip)

In [5]:
print("\n" + "="*60)
print("EJEMPLO 3: Ruta más corta con RAG (Gemini)")
print("="*60)

pregunta_ruta = """
Encontrar la ruta más corta desde el nodo A hasta el nodo E en el siguiente grafo no dirigido:
A - B: 4
A - C: 2
B - C: 1
B - D: 5
C - D: 8
C - E: 10
D - E: 2
"""

resultado_ruta = preguntar_io(pregunta_ruta, vectorstore)
print(resultado_ruta)

In [6]:
print("\n" + "="*60)
print("EJEMPLO 4: Problema de Redes con Gemini (Flujo máximo)")
print("="*60)

pregunta_redes = """
Encontrar el flujo máximo desde el nodo origen S hasta el nodo destino T en la siguiente red:

S -> A: capacidad 10
S -> B: capacidad 5
A -> B: capacidad 15
A -> C: capacidad 10
B -> C: capacidad 5
B -> T: capacidad 10
C -> T: capacidad 10
"""

resultado = preguntar_io(pregunta_redes, vectorstore)
print(resultado)

In [7]:
print("\n" + "="*60)
print("EJEMPLO 5: PERT/CPM con datos del problema")
print("="*60)

pregunta_pert = """
Un proyecto tiene las siguientes actividades:
A: duración 3 días, predecesores: ninguna
B: duración 5 días, predecesores: A
C: duración 2 días, predecesores: A
D: duración 4 días, predecesores: B, C

Calcular:
- Tiempo early y late de cada evento
- Holguras
- Ruta crítica
"""

resultado2 = preguntar_io(pregunta_pert, vectorstore)
print(resultado2)

---
## 🛠️ Utilidades

In [11]:
# ============================================================
# CONSULTA LIBRE: Escribe tu propio problema de IO
# ============================================================

mi_pregunta = """
Una empresa de muebles fabrica mesas y sillas. Cada mesa requiere 4 horas de carpintería 
y 2 horas de acabado. Cada silla requiere 3 horas de carpintería y 1 hora de acabado. 
Se dispone de 240 horas de carpintería y 100 horas de acabado por semana. 
La ganancia por mesa es de $70 y por silla es de $50.
Además, se deben producir al menos 10 sillas por semana por contrato con un distribuidor, 
y no se pueden producir más de 40 mesas por limitaciones de almacén.
Formular y resolver el problema de programación lineal para maximizar las ganancias semanales.
¿Cuál es la solución óptima, cuáles restricciones son activas, y cuál es la solucion para la maxima ganancia?
"""

resultado = preguntar_io(mi_pregunta, vectorstore)
print(resultado)

INFO     | io_calc.rag          | ============================================================
INFO     | io_calc.rag          | NUEVA CONSULTA: 
Una empresa de muebles fabrica mesas y sillas. Cada mesa requiere 4 horas de carpintería 
y 2 horas...
INFO     | io_calc.rag          | ============================================================
INFO     | io_calc.rag          | Buscando contexto relevante en ChromaDB...


🔍 Buscando contexto relevante...


INFO     | io_calc.embeddings   | Contexto encontrado — Páginas: [290, 107, 106, 931, 289], Scores: ['0.411', '0.443', '0.444', '0.453', '0.457']
INFO     | io_calc.rag          | --- Intentando modelo 1/2: gemini-2.5-flash ---


  📖 Contexto extraído de páginas: [290, 107, 106, 931, 289]
  📊 Scores de distancia: ['0.411', '0.443', '0.444', '0.453', '0.457']


INFO     | io_calc.rag          | Enviando prompt a gemini-2.5-flash (intento 1/3)...


🤖 Consultando gemini-2.5-flash...


INFO     | io_calc.rag          | ✅ Respuesta recibida de gemini-2.5-flash en 22.7s (6033 chars)
INFO     | io_calc.rag          | Respuesta guardada en caché (clave: f6f10f458a466ba3de62abfe90830cb4)


💾 Respuesta guardada en caché
¡Excelente! Como experto en Investigación de Operaciones, analizaré este problema de Programación Lineal paso a paso.

---

### 1. Clasificación del Problema

Este es un problema de **Programación Lineal (LP)**. Se busca optimizar (maximizar) una función objetivo lineal (ganancia) sujeta a un conjunto de restricciones lineales (recursos disponibles, requisitos de producción, límites de capacidad).

---

### 2. Formulación Matemática Completa

**Variables de Decisión:**
*   `x1`: Número de mesas a producir por semana.
*   `x2`: Número de sillas a producir por semana.

**Función Objetivo (Maximizar Ganancia):**
*   Cada mesa genera $70 de ganancia.
*   Cada silla genera $50 de ganancia.
*   `Maximizar Z = 70x1 + 50x2`

**Restricciones:**

1.  **Horas de Carpintería:**
    *   Cada mesa requiere 4 horas.
    *   Cada silla requiere 3 horas.
    *   Disponibilidad: 240 horas por semana.
    *   `4x1 + 3x2 <= 240`

2.  **Horas de Acabado:**
    *   Cada mesa re

In [9]:
# ============================================================
# UTILIDAD: Regenerar embeddings o limpiar cachés
# ============================================================

# Descomenta la acción que necesites:

# --- Regenerar ChromaDB (volver a procesar el PDF) ---
# from embeddings import delete_vectorstore
# delete_vectorstore()

# --- Limpiar caché de respuestas ---
# from rag import limpiar_cache_respuestas
# limpiar_cache_respuestas()